# Market Terminal im Notebook

Dieses Notebook holt seine Daten aus dem **laufenden Terminal** – das Terminal muss also gestartet sein.
Mit `Umschalt + Enter` führst du eine Zelle aus und springst zur nächsten.

- Ticker schreibst du wie in der Befehlszeile des Terminals: `SAP GR`, `SPY US`, `GOLD Comdty`, `DAX Index`.
- Ergebnisse kommen als Tabellen (pandas), die du weiterrechnen oder zeichnen kannst.
- Das Notebook **liest nur** – es ändert nichts im Terminal (keine Alarme, kein Depot).
- Neue Abfragen verbrauchen EODHD-Abrufe wie in der Oberfläche; was das Terminal schon geladen hat, kommt aus dem Zwischenspeicher.

Eigene Notebooks legst du links im Dateibereich mit **＋** an. Diese Datei ist das Beispiel – Änderungen hier bleiben gespeichert.

In [ ]:
import mt
import matplotlib.pyplot as plt
plt.style.use("dark_background")   # dunkel wie das Terminal; weglassen für helle Diagramme

mt.status()["mode"]   # "demo" = Beispieldaten, "eodhd" = echte Kurse

In [ ]:
mt.hilfe()   # alle Funktionen

## 1. Kurse und Verläufe

In [ ]:
# erster Wert deiner Watchlist – funktioniert mit Demo- und echten Daten
ticker = mt.status()["defaults"]["watchlist"][0]
kurse = mt.history(ticker, start="2025-01-01")
kurse.tail()

In [ ]:
kurse["close"].plot(title=f"{ticker} – Schlusskurs", figsize=(9, 3.5))
plt.show()

## 2. Mehrere Werte vergleichen

`mt.closes` legt Schlusskurse nebeneinander. Daraus lassen sich Wertentwicklung und Korrelationen rechnen.

In [ ]:
werte = ["SPY US", "GOLD Comdty", "WTI Comdty"]
preise = mt.closes(werte, start="2024-01-01").dropna()
(preise / preise.iloc[0] * 100).plot(title="Wertentwicklung (Start = 100)", figsize=(9, 3.5))
plt.show()

In [ ]:
renditen = preise.pct_change().dropna()
renditen.corr().round(2)   # Korrelation der Tagesrenditen

In [ ]:
# Schwankung p. a. (annualisiert) über die letzten 60 Handelstage
(renditen.rolling(60).std() * 252 ** 0.5 * 100).plot(title="Schwankung 60 Tage (% p. a.)", figsize=(9, 3.5))
plt.show()

## 3. Makro: Länderkennzahlen und Zinskurven

In [ ]:
inflation = mt.countries("Inflation")   # Jahreswerte, Spalten = Länder
inflation.tail(8)

In [ ]:
ax = inflation[["USA", "DEU"]].plot(title="Inflation (% gg. Vorjahr)", figsize=(9, 3.5), marker="o")
ax.xaxis.get_major_locator().set_params(integer=True)   # nur ganze Jahre an der Achse
plt.show()

In [ ]:
mt.curves(["US", "DE"])   # Rendite je Laufzeit in %

## 4. Prognose mit Einflussgrößen

Wie im Terminal (ECST): Inflation USA, zusätzlich mit Öl, Gold, S&P 500 und der 10-jährigen Rendite.
Das Urteil des Rückblick-Tests steht in `prognose.attrs`.

In [ ]:
prognose = mt.forecast("USA", "Inflation", x=["WTI", "Gold", "S&P 500", "US10Y.GBOND"])
print(prognose.attrs["verdict"])
print(prognose.attrs["features"])
prognose.tail(5)

In [ ]:
ist = prognose[prognose["kind"] == "Ist"]["value"]
fc = prognose[prognose["kind"] == "Prognose"]
ax = ist.plot(label="Ist", figsize=(9, 3.5))
fc["value"].plot(ax=ax, style="--", label="Prognose")
ax.fill_between(fc.index, fc["lo80"], fc["hi80"], alpha=0.2, label="80-%-Band")
if "ohne_einflussgroessen" in fc:
    fc["ohne_einflussgroessen"].plot(ax=ax, style=":", label="ohne Einflussgrößen")
ax.legend(); plt.show()

## 5. Depot und innerer Wert

In [ ]:
depot = mt.portfolio()
# pnlPctBase = Gewinn/Verlust seit Kauf in der Depotwährung (wie in PORT); pnlPct wäre in der Handelswährung der Aktie
depot[["name", "quantity", "price", "value", "weight", "pnlPctBase"]].round(2)

In [ ]:
d = mt.dcf(ticker)
print(d["name"], "– innerer Wert je Aktie:", round(d["valuePerShare"] or 0, 2), d["currency"], "| Kurs:", round(d["price"], 2))
# eigene Annahme: 5 Prozentpunkte mehr Wachstum im nächsten Jahr (Anteile, nicht Prozent)
anders = mt.dcf(ticker, levers={"growth_next": (d["levers"][0]["value"] + 0.05)})
round(anders["valuePerShare"] or 0, 2)

## 6. Alles andere

`mt.get(pfad, …)` ruft jede lesende Abfrage des Terminals auf, z. B. `mt.get("/macro/spreads", years=5)`.
Eine Übersicht aller Abfragen zeigt das Terminal unter http://127.0.0.1:8000/docs.